# Fashion Metadata + CLIP/FAISS Indexer

**Clean pipeline: Florence (captions) → spaCy (structured tags) → metadata JSON, then CLIP embeddings → FAISS index.**

## Section 1 — Setup

In [12]:
!pip install -q \
    open_clip_torch \
    faiss-gpu \
    timm \
    transformers==4.49.0 \
    tokenizers==0.21.0 \
    accelerate \
    sentencepiece \
    spacy
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 42.3 MB/s eta 0:00:0000:0100:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [13]:
import os
import json
import time
import random

import torch
import numpy as np
import pandas as pd
import faiss
import open_clip
import spacy

from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm
from transformers import AutoProcessor, AutoModelForCausalLM

In [14]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cuda


In [15]:
# Fashionpedia images live here on Kaggle
image_root = Path("/kaggle/input/datasets/riddhipandya1/images/test")

image_paths = sorted(image_root.glob("*.jpg"))

print("Total images found:", len(image_paths))
print(image_paths[:3])

Total images found: 3200
[PosixPath('/kaggle/input/datasets/riddhipandya1/images/test/003d41dd20f271d27219fe7ee6de727d.jpg'), PosixPath('/kaggle/input/datasets/riddhipandya1/images/test/0046f98599f05fd7233973e430d6d04d.jpg'), PosixPath('/kaggle/input/datasets/riddhipandya1/images/test/004e9e21cd1aca568a8ffc77a54638ce.jpg')]


In [16]:
inventory = pd.DataFrame({
    "image_id": [p.stem for p in image_paths],
    "image_path": [str(p) for p in image_paths]
})

inventory.to_csv("/kaggle/working/inventory.csv", index=False)

print("Inventory saved:", len(inventory), "images")
inventory.head()

Inventory saved: 3200 images


,image_id,image_path
0,003d41dd20f271d27219fe7ee6de727d,/kaggle/input/datasets/riddhipandya1/images/te...
1,0046f98599f05fd7233973e430d6d04d,/kaggle/input/datasets/riddhipandya1/images/te...
2,004e9e21cd1aca568a8ffc77a54638ce,/kaggle/input/datasets/riddhipandya1/images/te...
3,005b37fce3c0f641d327d95dd832f51b,/kaggle/input/datasets/riddhipandya1/images/te...
4,0094940c58c343b742f48ae26eb5e9fa,/kaggle/input/datasets/riddhipandya1/images/te...


## Section 2 — Florence (image captioning)

In [17]:
florence_path = "/kaggle/input/datasets/riddhipandya1/florence-official"

processor = AutoProcessor.from_pretrained(
    florence_path,
    trust_remote_code=True,
    local_files_only=True
)

model = AutoModelForCausalLM.from_pretrained(
    florence_path,
    trust_remote_code=True,
    local_files_only=True,
    torch_dtype=torch.float16
).to(device)

print("Florence loaded")

Florence loaded


In [18]:
SCENE_PROMPT = "<MORE_DETAILED_CAPTION>"

def generate_scene_description(image_path):

    image = Image.open(image_path).convert("RGB")

    inputs = processor(
        text=SCENE_PROMPT,
        images=image,
        return_tensors="pt"
    ).to(device, torch.float16)

    generated_ids = model.generate(
        input_ids=inputs["input_ids"],
        pixel_values=inputs["pixel_values"],
        max_new_tokens=128,
        do_sample=False
    )

    # Remove prompt tokens
    generated_ids = generated_ids[:, inputs["input_ids"].shape[1]:]

    description = processor.batch_decode(
        generated_ids,
        skip_special_tokens=True
    )[0]

    return description.strip()

## Section 3 — spaCy (structured feature extraction)

In [19]:
nlp = spacy.load("en_core_web_sm")
print("spaCy loaded")

spaCy loaded


In [20]:
def extract_spacy_features(caption):

    text = caption.lower()
    doc = nlp(text)

    # --------------------------------------------------
    # Vocabulary
    # NOTE: sets store BASE (lemma) forms. Matching checks both a
    # token's lemma and its raw text, so inflected forms like
    # "standing" / "stood" / "stands" or "sat" / "sitting" all match
    # the same base entry ("stand", "sit").
    # --------------------------------------------------

    COLORS = {
        "black","white","red","blue","green","yellow","orange","pink",
        "purple","brown","grey","gray","beige","cream","gold","silver",
        "maroon","navy","olive","teal","cyan","turquoise"
    }

    NECKLINES = {
        "crew neck","crew-neck","round neck","v-neck","v neck",
        "sweetheart","sweetheart neckline","halter","boat neck",
        "square neck","scoop neck","off shoulder","off-shoulder",
        "one shoulder","one-shoulder","high neck","turtleneck"
    }

    GARMENTS = {
        "shirt","t-shirt","tee","top","blouse","tank","dress","gown",
        "skirt","jean","pant","trouser","legging","short",
        "hoodie","coat","raincoat","jacket","blazer","cardigan","sweater",
        "kurta","saree","jumpsuit","romper","vest","suit"
    }

    HAIR_COLORS = {
        "black","brown","blonde","blond","grey","gray",
        "red","auburn","silver","white"
    }

    # base/lemma forms — "standing"/"stood"/"stands" all lemmatize to "stand"
    POSES = {
        "stand","walk","run",
        "sit","kneel","lean",
        "pose","look","face"
    }

    HAIR_STYLES = {
        "curly","straight","wavy","loose waves",
        "ponytail","bun","braid","bob"
    }

    FOOTWEAR = {
        "shoe","heel","boot",
        "sandal","sneaker","loafer","flat"
    }

    ACCESSORIES = {
        "bag","handbag","backpack","belt","watch",
        "hat","cap","necklace","bracelet","ring",
        "earring","scarf","glasses","sunglasses","tie"
    }

    PATTERNS = {
        "plain","solid","striped","checked","checkered",
        "plaid","floral","printed","graphic",
        "paisley","polka","animal","camouflage"
    }

    MATERIALS = {
        "cotton","denim","leather","silk","linen","lace",
        "knit","velvet","satin","polyester",
        "wool","chiffon","mesh"
    }

    SLEEVES = {
        "sleeveless","short-sleeve","long-sleeve",
        "half-sleeve","full-sleeve"
    }

    FITS = {
        "oversized","slim","loose","tight",
        "regular","relaxed","fitted"
    }

    LENGTHS = {
        "mini","midi","maxi","cropped","long"
    }

    # base/lemma forms — "standing outdoors"/"outdoor"/"outdoors" all normalize
    SETTINGS = {
        "indoor","outdoor","runway","fashion","street",
        "park","beach","studio","office","mall",
        "restaurant","home","stage","party","wedding","city"
    }

    ENVIRONMENT = {
        "tree","forest","grass","leaf",
        "flower","river","lake","water",
        "mountain","snow","road","street","beach",
        "park","building","sky","cloud","bush"
    }

    VIBES = {
        "casual","formal","sporty","business","professional",
        "vintage","elegant","party",
        "minimalist","bohemian","luxury",
        "streetwear"
    }

    # base/lemma forms — "smiling"/"smiled"/"smiles" all lemmatize to "smile"
    EXPRESSIONS = {
        "smile","laugh","happy","serious",
        "thoughtful","neutral","confident",
        "sad","angry","surprised"
    }

    LIGHTING = {
        "bright","soft","warm","natural",
        "dim","dramatic","sunlit","sunlight"
    }

    OBJECTS = {
        "tree","flower",
        "leaf","road","street",
        "building","camera","bench",
        "chair","table","river","lake",
        "water","person",
        "car","grass","sky","cloud"
    }

    # --------------------------------------------------
    # Output Dictionary
    # --------------------------------------------------

    features = {

        # Primary fashion attributes
        "gender": "",
        "age_group": "",

        "garments": [],
        "colors": [],
        "patterns": [],
        "materials": [],
        "necklines": [],
        "sleeves": [],
        "fit": [],
        "length": [],

        "footwear": [],
        "accessories": [],

        "hair_colors": [],
        "hair_styles": [],

        "expressions": [],
        "poses": [],

        # Scene information
        "setting": [],
        "environment": [],
        "objects": [],
        "lighting": [],
        "vibe": [],

        # Search keywords
        "keywords": [],

        # Per-garment bindings, e.g. a "red shirt and black pants" caption
        # produces [{"item": "shirt", "colors": ["red"], ...},
        #           {"item": "pant",  "colors": ["black"], ...}]
        # instead of flattening everything into one shared colors/garments bag.
        "garment_details": [],

        # NLP metadata (least important)
        "nouns": [],
        "adjectives": [],
        "verbs": [],
        "noun_chunks": []
    }

    # --------------------------------------------------
    # Token Extraction (checks BOTH lemma and raw text against
    # each vocab set, so inflected/tense variants aren't missed)
    # --------------------------------------------------

    for token in doc:

        lemma = token.lemma_.lower()
        raw = token.text.lower()
        forms = {lemma, raw}

        if token.pos_ in ["NOUN", "PROPN"]:
            features["nouns"].append(lemma)

        if token.pos_ == "VERB":
            features["verbs"].append(lemma)

        if token.pos_ == "ADJ":
            features["adjectives"].append(lemma)

        def add_if_match(vocab, bucket):
            hit = forms & vocab
            if hit:
                features[bucket].append(sorted(hit)[0])

        add_if_match(COLORS, "colors")
        add_if_match(GARMENTS, "garments")
        add_if_match(HAIR_COLORS, "hair_colors")
        add_if_match(HAIR_STYLES, "hair_styles")
        add_if_match(POSES, "poses")
        add_if_match(ENVIRONMENT, "environment")
        add_if_match(FOOTWEAR, "footwear")
        add_if_match(ACCESSORIES, "accessories")
        add_if_match(PATTERNS, "patterns")
        add_if_match(MATERIALS, "materials")
        add_if_match(SLEEVES, "sleeves")
        add_if_match(FITS, "fit")
        add_if_match(LENGTHS, "length")
        add_if_match(SETTINGS, "setting")
        add_if_match(VIBES, "vibe")
        add_if_match(EXPRESSIONS, "expressions")
        add_if_match(LIGHTING, "lighting")
        add_if_match(OBJECTS, "objects")

    # --------------------------------------------------
    # Multi-word Attribute Matching (genuinely multi-token terms only —
    # single-word terms are already covered by the lemma/text check above)
    # --------------------------------------------------

    for neckline in NECKLINES:
        if neckline in text:
            features["necklines"].append(neckline)

    for style in HAIR_STYLES:
        if " " in style and style in text:
            features["hair_styles"].append(style)

    # --------------------------------------------------
    # Gender
    # --------------------------------------------------

    if any(x in text for x in ["woman","female","girl","lady"," she "," her "]):
        features["gender"] = "female"

    elif any(x in text for x in ["man","male","boy","gentleman"," he "," his "]):
        features["gender"] = "male"

    # --------------------------------------------------
    # Age Group
    # --------------------------------------------------

    if any(x in text for x in ["child","kid","toddler"]):
        features["age_group"] = "child"

    elif any(x in text for x in ["teen","teenager"]):
        features["age_group"] = "teen"

    elif any(x in text for x in ["young","young adult"]):
        features["age_group"] = "young"

    elif "adult" in text:
        features["age_group"] = "adult"

    elif any(x in text for x in ["elderly","senior","old man","old woman"]):
        features["age_group"] = "elderly"

    # --------------------------------------------------
    # Per-Garment Attribute Binding
    #
    # For every token that IS a garment/footwear noun, walk its direct
    # dependency children (adjectival modifiers, compounds) to find the
    # color/pattern/material/fit/sleeve/length words that specifically
    # describe THAT garment — not just "any color mentioned anywhere in
    # the caption". This is what lets "red shirt and black pants" match
    # a query for "red shirt" without also matching "red pants".
    # --------------------------------------------------

    garment_vocab = GARMENTS | FOOTWEAR | ACCESSORIES
    seen_tokens = set()

    for token in doc:
        lemma = token.lemma_.lower()
        raw = token.text.lower()

        if not ((lemma in garment_vocab or raw in garment_vocab)):
            continue

        if token.i in seen_tokens:
            continue
        seen_tokens.add(token.i)

        item_colors, item_patterns, item_materials = [], [], []
        item_fit, item_sleeves, item_length = [], [], []

        # Modifiers directly attached to THIS garment token only.
        # (Deliberately not walking up to token.head's other children —
        # in "red shirt and black pants" the two garments are siblings
        # joined by "conj", and pulling in the head's children would
        # leak "red" from shirt onto pants. Direct children already
        # catch amod/compound modifiers correctly regardless of
        # whether this garment is the conjunction root or a conjunct.)
        for child in token.children:
            cforms = {child.lemma_.lower(), child.text.lower()}

            if child.dep_ not in ("amod", "compound", "conj", "nmod"):
                continue

            if cforms & COLORS:
                item_colors.append(sorted(cforms & COLORS)[0])
            if cforms & PATTERNS:
                item_patterns.append(sorted(cforms & PATTERNS)[0])
            if cforms & MATERIALS:
                item_materials.append(sorted(cforms & MATERIALS)[0])
            if cforms & FITS:
                item_fit.append(sorted(cforms & FITS)[0])
            if cforms & SLEEVES:
                item_sleeves.append(sorted(cforms & SLEEVES)[0])
            if cforms & LENGTHS:
                item_length.append(sorted(cforms & LENGTHS)[0])

            # handle "red and black shirt" — conjuncts of an amod
            for cc in child.children:
                if cc.dep_ == "conj":
                    ccforms = {cc.lemma_.lower(), cc.text.lower()}
                    if ccforms & COLORS:
                        item_colors.append(sorted(ccforms & COLORS)[0])
                    if ccforms & PATTERNS:
                        item_patterns.append(sorted(ccforms & PATTERNS)[0])
                    if ccforms & MATERIALS:
                        item_materials.append(sorted(ccforms & MATERIALS)[0])

        features["garment_details"].append({
            "item": lemma if lemma in garment_vocab else raw,
            "colors": sorted(set(item_colors)),
            "patterns": sorted(set(item_patterns)),
            "materials": sorted(set(item_materials)),
            "fit": sorted(set(item_fit)),
            "sleeves": sorted(set(item_sleeves)),
            "length": sorted(set(item_length)),
        })

    # --------------------------------------------------
    # Noun Chunks
    # --------------------------------------------------

    features["noun_chunks"] = [chunk.text.lower() for chunk in doc.noun_chunks]

    # --------------------------------------------------
    # Remove Duplicates
    # --------------------------------------------------

    for key in features:
        if isinstance(features[key], list) and key != "garment_details":
            features[key] = sorted(set(features[key]))

    # --------------------------------------------------
    # Search Keywords
    # --------------------------------------------------

    features["keywords"] = sorted(set(
        features["colors"] +
        features["garments"] +
        features["necklines"] +
        features["hair_colors"] +
        features["hair_styles"] +
        features["poses"] +
        features["environment"] +
        features["footwear"] +
        features["accessories"] +
        features["patterns"] +
        features["materials"] +
        features["sleeves"] +
        features["fit"] +
        features["length"] +
        features["setting"] +
        features["vibe"] +
        features["expressions"] +
        features["lighting"] +
        features["objects"]
    ))

    return features

## Section 4 — Test Pipeline
Run Florence → spaCy on 5 random images to sanity-check before the full run.

In [21]:
from pprint import pprint

test_df = inventory.sample(5, random_state=42).reset_index(drop=True)

for idx, row in test_df.iterrows():

    image_id = row["image_id"]
    image_path = row["image_path"]

    caption = generate_scene_description(image_path)
    metadata = extract_spacy_features(caption)

    print("=" * 100)
    print(f"Image {idx+1}/5 | {image_id}")
    print("\nFLORENCE CAPTION:\n", caption)
    print("\nSPACY FEATURES:\n")
    pprint(metadata, sort_dicts=False)
    print("=" * 100, "\n")

Image 1/5 | bedd4c227a48bc3f89431058fa77c82b

FLORENCE CAPTION:
 . She is wearing a strapless dress with a colorful floral print in shades of blue, green, and pink. The dress has a sweetheart neckline and a flowy skirt that falls below her knees. She has blonde hair styled in loose waves and is wearing yellow rain boots. The woman is smiling and looking at the camera. In the background, there are trees and people walking around. The ground is covered in fallen leaves and there is a tree trunk on the left side of the image.

SPACY FEATURES:

{'gender': 'female',
 'age_group': '',
 'garments': ['dress', 'skirt'],
 'colors': ['blue', 'green', 'pink', 'yellow'],
 'patterns': ['floral'],
 'materials': [],
 'necklines': ['sweetheart', 'sweetheart neckline'],
 'sleeves': [],
 'fit': ['loose'],
 'length': [],
 'footwear': ['boot'],
 'accessories': [],
 'hair_colors': ['blonde'],
 'hair_styles': ['loose waves'],
 'expressions': ['smile'],
 'poses': ['look', 'walk'],
 'setting': [],
 'environmen

## Section 5 — Build Fashion Metadata Index
Run Florence → spaCy over 1500 images and save to `fashion_metadata_1500.json`.

In [22]:
NUM_IMAGES = 1500

# Same random 1500 images every run
sample_df = inventory.sample(NUM_IMAGES, random_state=42).reset_index(drop=True)

results = []
start = time.time()

for idx, row in sample_df.iterrows():

    image_id = row["image_id"]
    image_path = row["image_path"]

    try:
        description = generate_scene_description(image_path)
        metadata = extract_spacy_features(description)

        results.append({
            "image_id": image_id,
            "image_path": image_path,
            "description": description,
            **metadata
        })

        print(f"Processed {idx+1}/{NUM_IMAGES}")

    except Exception as e:
        print(f"Error : {image_id}")
        results.append({
            "image_id": image_id,
            "image_path": image_path,
            "description": "",
            "error": str(e)
        })

    # Backup every 100 images
    if (idx + 1) % 100 == 0:
        with open("/kaggle/working/fashion_metadata_1500.json", "w") as f:
            json.dump(results, f, indent=4)
        print("Backup saved")

# Final save
with open("/kaggle/working/fashion_metadata_1500.json", "w") as f:
    json.dump(results, f, indent=4)

print("\nFinished!")
print(f"Processed : {len(results)} images")
print(f"Total time : {(time.time() - start) / 60:.2f} minutes")

Processed 1/1500
Processed 2/1500
Processed 3/1500
Processed 4/1500
Processed 5/1500
Processed 6/1500
Processed 7/1500
Processed 8/1500
Processed 9/1500
Processed 10/1500
Processed 11/1500
Processed 12/1500
Processed 13/1500
Processed 14/1500
Processed 15/1500
Processed 16/1500
Processed 17/1500
Processed 18/1500
Processed 19/1500
Processed 20/1500
Processed 21/1500
Processed 22/1500
Processed 23/1500
Processed 24/1500
Processed 25/1500
Processed 26/1500
Processed 27/1500
Processed 28/1500
Processed 29/1500
Processed 30/1500
Processed 31/1500
Processed 32/1500
Processed 33/1500
Processed 34/1500
Processed 35/1500
Processed 36/1500
Processed 37/1500
Processed 38/1500
Processed 39/1500
Processed 40/1500
Processed 41/1500
Processed 42/1500
Processed 43/1500
Processed 44/1500
Processed 45/1500
Processed 46/1500
Processed 47/1500
Processed 48/1500
Processed 49/1500
Processed 50/1500
Processed 51/1500
Processed 52/1500
Processed 53/1500
Processed 54/1500
Processed 55/1500
Processed 56/1500
P

## Section 6 — Load CLIP

In [23]:
weights_path = "/kaggle/input/datasets/riddhipandya1/clip-tensors-new/open_clip_model.safetensors"

clip_model, _, clip_preprocess = open_clip.create_model_and_transforms(
    "ViT-B-32",
    pretrained=weights_path
)

clip_tokenizer = open_clip.get_tokenizer("ViT-B-32")

clip_model = clip_model.to(device)
clip_model.eval()

print("CLIP loaded")

CLIP loaded


## Section 7 — Generate Image Embeddings
Loop over the same 1500 images used for the metadata index.

In [24]:
from torch.utils.data import Dataset, DataLoader

class FashionDataset(Dataset):

    def __init__(self, dataframe, preprocess):
        self.df = dataframe
        self.preprocess = preprocess

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        image = Image.open(self.df.iloc[idx]["image_path"]).convert("RGB")
        image = self.preprocess(image)
        image_id = self.df.iloc[idx]["image_id"]
        return {"image": image, "image_id": image_id}


dataset = FashionDataset(sample_df, clip_preprocess)

loader = DataLoader(
    dataset,
    batch_size=64,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("Total batches:", len(loader))

Total batches: 24


In [25]:
image_embeddings = []
image_ids = []

with torch.no_grad():
    for batch in tqdm(loader):

        images = batch["image"].to(device)
        features = clip_model.encode_image(images)
        features = features / features.norm(dim=-1, keepdim=True)

        image_embeddings.append(features.cpu())
        image_ids.extend(batch["image_id"])

image_embeddings = torch.cat(image_embeddings)

print(image_embeddings.shape)

  0%|          | 0/24 [00:00<?, ?it/s]

torch.Size([1500, 512])


In [26]:
np.save("/kaggle/working/clip_embeddings.npy", image_embeddings.numpy())
print("Embeddings saved")

Embeddings saved


## Section 8 — Build FAISS Index

In [27]:
embeddings_np = image_embeddings.numpy().astype("float32")

index = faiss.IndexFlatIP(512)
index.add(embeddings_np)

faiss.write_index(index, "/kaggle/working/faiss.index")

print("Indexed images:", index.ntotal)

Indexed images: 1500
